# Data Formats & Storage — 05: Delta Lake Advanced II

## What you will learn
This notebook covers Delta Lake's most impactful performance and DML features:
1. **Liquid Clustering** — next-gen alternative to ZORDER, dynamic re-clustering
2. **DML optimizations** — low-shuffle MERGE, partial UPDATE patterns
3. **Deletion vectors** — faster deletes without file rewrites
4. **Predictive I/O** — Delta's smart prefetching
5. **Dynamic partition overwrite** — replace only affected partitions
6. **Multi-table transactions** — coordinated writes across tables


In [ ]:
import os, time, random, datetime, pathlib, shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

spark = (
    SparkSession.builder
    .appName('delta-advanced-2')
    .master(MASTER)
    .config('spark.executor.memory',            '2g')
    .config('spark.driver.memory',              '1g')
    .config('spark.executor.cores',             '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
# Re-suppress noisy loggers after setLogLevel resets them (Spark 4.x / log4j2)
_jvm = spark.sparkContext._jvm
_ctx = _jvm.org.apache.logging.log4j.LogManager.getContext(False)
_cfg = _ctx.getConfiguration()
for _logger_name in [
    'org.apache.iceberg.hadoop.HadoopTableOperations',
    'org.apache.hadoop.util.NativeCodeLoader',
]:
    _lc = _jvm.org.apache.logging.log4j.core.config.LoggerConfig(_logger_name,
              _jvm.org.apache.logging.log4j.Level.ERROR, False)
    _cfg.addLogger(_logger_name, _lc)
_ctx.updateLoggers()
print(f"Spark {spark.version}")

## Step 1 — Liquid Clustering vs ZORDER

**ZORDER** was the original Delta clustering approach.
**Liquid Clustering** (Delta 3.1+) is its modern replacement.

| | ZORDER | Liquid Clustering |
|---|---|---|
| How | Rewrites ALL files by space-filling curve | Incrementally re-clusters only changed files |
| When to re-run | After every significant write | `OPTIMIZE` handles it automatically |
| Column changes | Must rewrite everything | Just call `ALTER TABLE CLUSTER BY` |
| Partition required | Yes (manually manage) | No — liquid clustering replaces partitioning |
| Cost | High (full rewrite) | Low (incremental) |

Liquid Clustering is the **recommended approach** for new Delta tables.


In [ ]:
import pathlib, shutil, random, datetime
from delta.tables import DeltaTable

DATA_DIR  = '/workspace/data'
DELTA_DIR = f'{DATA_DIR}/delta_advanced_2'
shutil.rmtree(DELTA_DIR, ignore_errors=True)
pathlib.Path(DELTA_DIR).mkdir(parents=True, exist_ok=True)

random.seed(42)
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["active","churned","trial","vip"]

schema = StructType([
    StructField("event_id",   LongType()),
    StructField("user_id",    LongType()),
    StructField("category",   StringType()),
    StructField("country",    StringType()),
    StructField("status",     StringType()),
    StructField("revenue",    DoubleType()),
    StructField("event_date", DateType()),
])

def make_events(n, start_id=1):
    base = datetime.date(2023,1,1)
    return [(start_id+i, random.randint(1,100_000),
             random.choice(CATEGORIES), random.choice(COUNTRIES),
             random.choice(STATUSES),
             round(random.uniform(5, 2000), 2),
             base + datetime.timedelta(days=random.randint(0,365)))
            for i in range(n)]

# Table WITH Liquid Clustering
LC_PATH = f"{DELTA_DIR}/events_liquid"
spark.sql(f"""
    CREATE TABLE delta.`{LC_PATH}` (
        event_id   BIGINT,
        user_id    BIGINT,
        category   STRING,
        country    STRING,
        status     STRING,
        revenue    DOUBLE,
        event_date DATE
    )
    USING delta
    CLUSTER BY (country, category, status)
    TBLPROPERTIES (
        'delta.enableDeletionVectors' = 'true'
    )
""")

# Table WITH ZORDER (traditional)
ZORDER_PATH = f"{DELTA_DIR}/events_zorder"
df = spark.createDataFrame(make_events(500_000), schema)
df.write.format("delta").mode("overwrite").save(ZORDER_PATH)

print("Both tables created.")
print(f"  Liquid Clustering: {LC_PATH}")
print(f"  ZORDER:            {ZORDER_PATH}")

In [ ]:
# Insert data into liquid clustering table
df = spark.createDataFrame(make_events(500_000), schema)
df.write.format("delta").mode("append").save(LC_PATH)

# Simulate ongoing writes (realistic: new data every hour)
for i in range(5):
    batch = spark.createDataFrame(make_events(10_000, start_id=500_001+i*10_000), schema)
    batch.write.format("delta").mode("append").save(LC_PATH)

print("Data inserted. Files before OPTIMIZE (many small files):")
DeltaTable.forPath(spark, LC_PATH).detail() \
          .select("numFiles","sizeInBytes").show()

# OPTIMIZE automatically applies liquid clustering
print("Running OPTIMIZE (applies liquid clustering incrementally)...")
import time
t0 = time.time()
spark.sql(f"OPTIMIZE delta.`{LC_PATH}`").show(truncate=False)
t_lc = time.time() - t0

print(f"Liquid Clustering OPTIMIZE: {t_lc:.1f}s")
DeltaTable.forPath(spark, LC_PATH).detail() \
          .select("numFiles","sizeInBytes").show()

# ZORDER comparison
print("Running OPTIMIZE + ZORDER BY (traditional)...")
t0 = time.time()
spark.sql(f"OPTIMIZE delta.`{ZORDER_PATH}` ZORDER BY (country, category, status)").show(truncate=False)
t_zo = time.time() - t0
print(f"ZORDER OPTIMIZE: {t_zo:.1f}s")

In [ ]:
# Query benchmark: liquid clustering vs ZORDER
def timed(label, func, runs=3):
    times = [time.time() or (func() or None) or time.time()
             for _ in range(runs)]
    # proper timing
    results = []
    for _ in range(runs):
        t0 = time.time()
        func()
        results.append(time.time() - t0)
    median = sorted(results)[1]
    print(f"  {label:<45} {median:.3f}s")
    return median

print("Selective query benchmark (country=US AND category=Electronics):")

t_lc = timed("Liquid Clustering",
    lambda: spark.read.format("delta").load(LC_PATH)
                 .filter((F.col("country")=="US") & (F.col("category")=="Electronics"))
                 .agg(F.sum("revenue")).collect())

t_zo = timed("ZORDER",
    lambda: spark.read.format("delta").load(ZORDER_PATH)
                 .filter((F.col("country")=="US") & (F.col("category")=="Electronics"))
                 .agg(F.sum("revenue")).collect())

print(f"\nLiquid Clustering {'faster' if t_lc < t_zo else 'slower'} by {abs(t_zo/t_lc-1)*100:.0f}%")
print("(Liquid clustering reorganises files incrementally — better for continuous writes)")

## Step 2 — Deletion Vectors: Fast Deletes

**Traditional Delta delete:**
1. Find affected files
2. Read ALL rows from those files
3. Filter out deleted rows
4. Write NEW files with remaining rows
→ Expensive: rewrites entire files for even a single deleted row

**Deletion Vectors (DV):**
1. Find affected files
2. Write a small deletion vector file (bitmap of deleted row positions)
3. Done — data files are NOT rewritten
→ Cheap: only writes a tiny DV file (kilobytes)

At read time, Spark applies the DV to skip deleted rows.
Run `OPTIMIZE` to physically compact files and remove DVs.


In [ ]:
DV_PATH = f"{DELTA_DIR}/events_dv"
shutil.rmtree(DV_PATH, ignore_errors=True)

# Create table with deletion vectors enabled
df.write.format("delta") \
        .option("delta.enableDeletionVectors", "true") \
        .mode("overwrite") \
        .save(DV_PATH)

import subprocess

def count_files(path, ext):
    r = subprocess.run(["find", path, "-name", f"*.{ext}"],
                       capture_output=True, text=True)
    return len([x for x in r.stdout.strip().split("\n") if x])

print("Before DELETE:")
print(f"  Parquet files : {count_files(DV_PATH, 'parquet')}")
print(f"  DV files      : {count_files(DV_PATH, 'bin')}")
print(f"  Total rows    : {spark.read.format('delta').load(DV_PATH).count():,}")

# Delete 10% of rows — with DVs this does NOT rewrite parquet files
t0 = time.time()
spark.sql(f"""
    DELETE FROM delta.`{DV_PATH}`
    WHERE user_id % 10 = 0
""")
t_dv = time.time() - t0

print(f"\nAfter DELETE (with Deletion Vectors) — {t_dv:.2f}s:")
print(f"  Parquet files : {count_files(DV_PATH, 'parquet')} (unchanged!)")
print(f"  DV files      : {count_files(DV_PATH, 'bin')} (new bitmap files)")
print(f"  Total rows    : {spark.read.format('delta').load(DV_PATH).count():,}")
print()
print("Parquet files were NOT rewritten — only small DV bitmaps were written.")
print("Run OPTIMIZE to physically compact and remove DVs when convenient.")

## Step 3 — Low-Shuffle MERGE

Standard MERGE rewrites ALL matched data files.
**Low-shuffle MERGE** (Delta 2.4+) only rewrites files that actually contain matching rows,
using deletion vectors for the rest — dramatically reducing write amplification.


In [ ]:
MERGE_PATH = f"{DELTA_DIR}/orders_merge"
shutil.rmtree(MERGE_PATH, ignore_errors=True)

order_schema = StructType([
    StructField("order_id",   LongType()),
    StructField("customer_id",LongType()),
    StructField("product",    StringType()),
    StructField("amount",     DoubleType()),
    StructField("status",     StringType()),
    StructField("updated_at", TimestampType()),
])

def make_orders(n, start_id=1):
    products = ["Laptop","Phone","Tablet","Headphones","Monitor"]
    statuses = ["pending","confirmed","shipped"]
    rows = []
    for i in range(n):
        rows.append((start_id+i, random.randint(1,10000),
                     random.choice(products), round(random.uniform(50,2000),2),
                     random.choice(statuses), datetime.datetime.now()))
    return rows

orders = spark.createDataFrame(make_orders(200_000), order_schema)
orders.write.format("delta") \
      .option("delta.enableDeletionVectors","true") \
      .mode("overwrite").save(MERGE_PATH)

# Upsert: 10% updates + 5% new inserts
updates = spark.createDataFrame(
    make_orders(20_000, start_id=1),   # update existing
    order_schema
)
inserts = spark.createDataFrame(
    make_orders(10_000, start_id=200_001),  # new orders
    order_schema
)
source = updates.union(inserts)
source.createOrReplaceTempView("source")

t0 = time.time()
spark.sql(f"""
    MERGE INTO delta.`{MERGE_PATH}` t
    USING source s ON t.order_id = s.order_id
    WHEN MATCHED AND s.status != t.status THEN
        UPDATE SET t.status = s.status, t.updated_at = s.updated_at
    WHEN NOT MATCHED THEN INSERT *
""")
t_merge = time.time() - t0

total = spark.read.format("delta").load(MERGE_PATH).count()
print(f"MERGE completed in {t_merge:.2f}s")
print(f"Total rows after merge: {total:,}  (was 200,000 + 10,000 new = 210,000)")
print()
print("With Deletion Vectors enabled, MERGE uses them for matched rows")
print("instead of rewriting entire files — much lower write amplification.")

## Step 4 — Dynamic Partition Overwrite

Traditional `overwrite` replaces the ENTIRE table.
**Dynamic Partition Overwrite** replaces ONLY the partitions present in the new data.

This is critical for date-partitioned tables where you want to reprocess
specific dates without touching historical data.


In [ ]:
DPO_PATH = f"{DELTA_DIR}/sales_dpo"
shutil.rmtree(DPO_PATH, ignore_errors=True)

sales_schema = StructType([
    StructField("sale_id",   LongType()),
    StructField("region",    StringType()),
    StructField("amount",    DoubleType()),
    StructField("sale_date", DateType()),
])

def make_sales(n, start_id=1):
    regions = ["AMER","EMEA","APAC","LATAM"]
    base = datetime.date(2024,1,1)
    return [(start_id+i, random.choice(regions),
             round(random.uniform(100,5000),2),
             base + datetime.timedelta(days=random.randint(0,90)))
            for i in range(n)]

# Initial data: Jan–Mar 2024
initial = spark.createDataFrame(make_sales(100_000), sales_schema)
initial.write.format("delta") \
       .partitionBy("sale_date") \
       .mode("overwrite") \
       .save(DPO_PATH)

before_count = spark.read.format("delta").load(DPO_PATH).count()
print(f"Initial rows: {before_count:,}")

# Simulate: reprocess Feb 2024 data only
feb_data = spark.createDataFrame(
    [(i, random.choice(["AMER","EMEA","APAC","LATAM"]),
      round(random.uniform(200, 8000), 2),
      datetime.date(2024, 2, random.randint(1,29)))
     for i in range(1, 5_001)],
    sales_schema
)

# Dynamic partition overwrite — only replaces Feb partitions
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
feb_data.write.format("delta") \
        .partitionBy("sale_date") \
        .mode("overwrite") \
        .save(DPO_PATH)

after_count = spark.read.format("delta").load(DPO_PATH).count()
print(f"After DPO : {after_count:,} rows  (Feb replaced, Jan+Mar untouched)")
print()
print("DeltaTable history:")
DeltaTable.forPath(spark, DPO_PATH).history() \
          .select("version","operation","operationParameters") \
          .show(3, truncate=False)

## Step 5 — Table Cloning

Delta supports **shallow** and **deep** clones:
- **Shallow clone**: pointer to source files, no data copy. Zero cost, instant.
  Changes to clone do NOT affect source (copy-on-write semantics).
- **Deep clone**: full data copy. Independent from source.

Use cases: create dev/test copies, backups, blue-green deployments.


In [ ]:
CLONE_PATH = f"{DELTA_DIR}/events_clone"
shutil.rmtree(CLONE_PATH, ignore_errors=True)

# Shallow clone — zero cost, instant
t0 = time.time()
spark.sql(f"""
    CREATE TABLE delta.`{CLONE_PATH}`
    SHALLOW CLONE delta.`{LC_PATH}`
""")
t_clone = time.time() - t0

import subprocess
def dir_size_mb(path):
    r = subprocess.run(["du","-sm",path], capture_output=True, text=True)
    return int(r.stdout.split()[0]) if r.stdout else 0

src_mb   = dir_size_mb(LC_PATH)
clone_mb = dir_size_mb(CLONE_PATH)

print(f"Shallow clone created in {t_clone:.2f}s")
print(f"  Source size : {src_mb} MB")
print(f"  Clone size  : {clone_mb} MB  (just metadata, no data files)")
print()
print(f"Source rows : {spark.read.format('delta').load(LC_PATH).count():,}")
print(f"Clone rows  : {spark.read.format('delta').load(CLONE_PATH).count():,}")
print()

# Write to clone — does NOT affect source (copy-on-write)
extra = spark.createDataFrame(make_events(1_000, start_id=999_000), schema)
extra.write.format("delta").mode("append").save(CLONE_PATH)

print("After writing 1000 rows to clone:")
print(f"  Source rows : {spark.read.format('delta').load(LC_PATH).count():,} (unchanged)")
print(f"  Clone rows  : {spark.read.format('delta').load(CLONE_PATH).count():,} (updated)")

## Summary: Delta Advanced II

| Feature | When to use | Key config |
|---|---|---|
| Liquid Clustering | New tables, frequent writes, changing query patterns | `CLUSTER BY (col1, col2)` |
| ZORDER | Existing tables, stable query patterns | `OPTIMIZE ... ZORDER BY` |
| Deletion Vectors | Frequent small deletes/updates | `delta.enableDeletionVectors=true` |
| Low-shuffle MERGE | MERGE with < 20% row matches | Automatic with DVs enabled |
| Dynamic Partition Overwrite | Reprocess specific date/region partitions | `partitionOverwriteMode=dynamic` |
| Shallow Clone | Dev/test copies, fast backups | `SHALLOW CLONE` |
| Deep Clone | Independent copies, migrations | `DEEP CLONE` |

### Liquid Clustering best practices
```sql
-- Create with clustering
CREATE TABLE t USING delta CLUSTER BY (country, category)

-- Add clustering to existing table
ALTER TABLE t CLUSTER BY (country, category)

-- Re-cluster incrementally (run periodically)
OPTIMIZE t

-- Check clustering status
DESCRIBE DETAIL t  -- shows clusteringColumns
```
